In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

X = pd.read_pickle(f"../input.pkl")
y = pd.read_pickle(f"../output.pkl")
newdf=pd.concat([X,y],axis=1)

In [2]:
newdf

,gender,age,race,educ,marry,house,pov,wt,ht,bmi,...,ldl,hdl,acratio,glu,insulin,crp,hb1ac,mvpa,ac_week,dbs
0,1.0,66.0,3.0,5.0,1.0,2.0,5.00,101.8,174.2,33.5,...,135.0,60.0,6.64,99.0,19.91,2.03,5.6,450.000000,17.307692,0.0
1,1.0,34.0,1.0,4.0,1.0,3.0,1.33,90.6,173.3,30.2,...,111.0,46.0,4.07,100.0,11.38,1.05,5.1,43.808976,4.000000,0.0
2,1.0,51.0,3.0,5.0,1.0,4.0,5.00,76.7,177.3,24.4,...,120.0,48.0,6.29,88.0,7.20,0.92,4.8,555.000000,0.576923,0.0
3,1.0,47.0,1.0,3.0,1.0,2.0,1.67,73.1,159.5,28.7,...,122.0,41.0,12.41,107.0,9.81,5.22,5.8,844.615385,3.500000,0.0
4,1.0,50.0,3.0,5.0,1.0,2.0,5.00,94.0,171.7,31.9,...,65.0,49.0,4.94,104.0,7.26,0.71,5.3,1260.000000,2.000000,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
758,1.0,68.0,3.0,4.0,1.0,4.0,1.30,94.8,178.6,29.7,...,150.0,47.0,46.56,99.0,16.61,3.69,5.4,1540.000000,0.173077,0.0
759,2.0,70.0,3.0,5.0,2.0,1.0,5.00,42.8,155.4,17.7,...,133.0,81.0,8.00,82.0,1.35,0.66,5.4,260.000000,1.000000,0.0
760,2.0,74.0,3.0,5.0,2.0,1.0,2.74,68.5,158.5,27.3,...,83.0,74.0,7.29,97.0,9.31,0.67,5.4,447.692308,4.000000,0.0
761,1.0,57.0,3.0,5.0,1.0,3.0,4.12,74.4,176.9,23.8,...,171.0,55.0,4.55,105.0,6.60,0.42,5.2,300.000000,7.000000,0.0


In [3]:
newdf[newdf['dbs']==2]

,gender,age,race,educ,marry,house,pov,wt,ht,bmi,...,ldl,hdl,acratio,glu,insulin,crp,hb1ac,mvpa,ac_week,dbs
9,1.0,57.0,2.0,2.0,1.0,2.0,2.73,75.1,161.0,29.0,...,101.0,53.0,5.01,163.0,22.65,0.42,6.6,330.000000,21.000000,2.0
43,2.0,67.0,2.0,3.0,2.0,1.0,4.05,59.4,156.2,24.3,...,86.0,58.0,8.69,100.0,7.33,0.71,6.4,1260.000000,0.028846,2.0
59,1.0,66.0,3.0,4.0,1.0,4.0,5.00,83.1,176.8,26.6,...,185.0,41.0,9.60,131.0,11.50,0.93,6.8,33.808976,4.000000,2.0
97,1.0,76.0,3.0,3.0,1.0,2.0,1.64,74.3,167.6,26.5,...,68.0,51.0,24.56,242.0,7.90,0.55,7.8,28.808976,0.173077,2.0
109,1.0,55.0,3.0,5.0,2.0,1.0,5.00,93.6,183.0,27.9,...,99.0,57.0,17.12,186.0,7.01,5.71,8.0,315.000000,10.500000,2.0
116,1.0,78.0,4.0,5.0,1.0,2.0,2.06,70.4,174.9,23.0,...,54.0,72.0,270.32,125.0,5.99,2.58,6.4,90.000000,0.259615,2.0
121,2.0,58.0,3.0,5.0,2.0,1.0,5.00,87.8,170.3,30.3,...,119.0,66.0,10.85,100.0,14.55,1.90,5.4,117.617952,6.000000,2.0
122,2.0,40.0,3.0,4.0,2.0,2.0,3.60,64.5,160.7,25.0,...,49.0,76.0,9.97,171.0,18.00,0.92,7.7,57.543596,34.615385,2.0
132,1.0,61.0,2.0,5.0,1.0,4.0,5.00,63.6,171.8,21.5,...,76.0,36.0,7.25,147.0,4.54,0.19,7.2,600.000000,0.576923,2.0
159,1.0,79.0,3.0,5.0,1.0,2.0,4.92,97.6,179.5,30.3,...,58.0,50.0,24.37,182.0,35.49,2.61,5.6,390.471807,2.000000,2.0


In [4]:
col_minmax = {
    idx: (newdf.iloc[:, idx].min(), newdf.iloc[:, idx].max())
    for idx in range(newdf.shape[1])
}
col_minmax


{0: (1.0, 2.0),
 1: (20.0, 80.0),
 2: (1.0, 7.0),
 3: (1.0, 5.0),
 4: (1.0, 3.0),
 5: (1.0, 7.0),
 6: (5.397605346934028e-79, 5.0),
 7: (42.8, 175.8),
 8: (140.3, 196.6),
 9: (17.5, 56.7),
 10: (63.7, 157.4),
 11: (77.8, 157.2),
 12: (38.0, 134.0),
 13: (36.0, 111.0),
 14: (87.0, 198.0),
 15: (5.0, 148.0),
 16: (2.8, 5.3),
 17: (9.0, 103.0),
 18: (0.39, 1.71),
 19: (96.0, 309.0),
 20: (24.0, 379.0),
 21: (5.0, 191.0),
 22: (2.2, 13.5),
 23: (9.8, 17.7),
 24: (30.6, 51.6),
 25: (30.0, 200.0),
 26: (23.0, 112.0),
 27: (0.22, 270.32),
 28: (59.0, 325.0),
 29: (0.35, 96.13),
 30: (0.11, 46.18),
 31: (3.8, 11.6),
 32: (4.423076923076923, 1920.0),
 33: (0.028846153846153848, 56.15384615384615),
 34: (0.0, 2.0)}

In [5]:
newdf.shape[0]

763

In [6]:
X.iloc[23]

gender       2.000000
age         68.000000
race         4.000000
educ         4.000000
marry        2.000000
house        1.000000
pov          2.130000
wt          77.400000
ht         164.800000
bmi         28.500000
wst        101.300000
hip        102.700000
dia         71.000000
pulse       90.000000
sys        131.000000
alt         26.000000
albumin      3.900000
ast         25.000000
crea         0.700000
chol       125.000000
tyg        192.000000
ggt         21.000000
wbc          6.100000
hb          13.300000
hct         39.800000
ldl         58.000000
hdl         35.000000
acratio      1.000000
glu        116.000000
insulin     25.110000
crp         10.820000
hb1ac        6.600000
mvpa       660.000000
ac_week      0.086538
Name: 23, dtype: float64

In [7]:
X[['alt']].max()

alt    148.0
dtype: float64

In [8]:
import torch.cuda
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

True
NVIDIA GeForce RTX 4070 SUPER


In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader,Subset
from lightning.pytorch import LightningModule,Trainer
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.model_selection import KFold
from lightning.pytorch.loggers import CSVLogger
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix


# 머신러닝 관련 라이브러리
from sklearn.metrics import classification_report, confusion_matrix

#기억력 관련 변수 ->cerad 4,6,7,8(8,10,11,12)
#사고력 관련 변수 ->cerad 1,2,3,5,9,10,11,12 (5,6,7,9,13,14,15,16)
# 나머지 변수 -> age,gender, edu,s_adl,s_iadl  (0,1,2,3,4)
# 범주형 파생변수 -> j1v, 2,5,6,9,10,11,adl,iadl


def to_tensor(*dfs):
    return [torch.tensor(df.values,dtype=torch.float32) for df in dfs]



class MainModule(nn.Module):
    def __init__(self,input_dim=6,output_dim=3,do1=0.5,activ=0):
        super().__init__()
        self.activ=nn.GELU() if activ==1 else nn.ReLU()
        self.input_layer=nn.Sequential(
            nn.Linear(input_dim,6),
            self.activ,
            nn.Dropout(do1),
            
            nn.Linear(6,4),
            self.activ,
            nn.Dropout(do1),

            nn.Linear(4,3),
            self.activ,
            nn.Dropout(do1),
        )
        
        self.output_layer=nn.Linear(3,output_dim)
    def forward(self,x):
        x=self.input_layer(x)
        output=self.output_layer(x)
        return output

class SubModule(nn.Module):
    def __init__(self,input_dim=28,output_dim=7,do1=0.5,activ=0):
        super().__init__()
        self.activ=nn.GELU() if activ==1 else nn.ReLU()
        self.input_layer=nn.Sequential(
            nn.Linear(input_dim,28),
            self.activ,
            nn.Dropout(do1),
            
            nn.Linear(28,14),
            self.activ,
            nn.Dropout(do1),

            nn.Linear(14,7),
            self.activ,
            nn.Dropout(do1),
        )
        
        self.output_layer=nn.Linear(7,output_dim)
    def forward(self,x):
        x=self.input_layer(x)
        output=self.output_layer(x)
        return output

class TotalModel(nn.Module):    #TotalModel의 128 dim에서 skip connection 적용
    def __init__(self,do1,do2,activ):
        super().__init__()

        self.activ=nn.GELU() if activ==1 else nn.ReLU()

        self.module1=SubModule(6,3,do1,activ)
        self.module2=SubModule(28,7,do1,activ)

        
        self.hidden_layers=nn.ModuleList([
            nn.Sequential(
                nn.Linear(10,10),
                self.activ,
                nn.Dropout(do2),
            )for _ in range(1)
        ])
        
        self.output_layers=nn.Sequential(
            nn.Linear(10,5),
            self.activ,
            nn.Dropout(do2),
            
            nn.Linear(5,5),
            self.activ,
            nn.Dropout(do2),            
            
            nn.Linear(5,3)         
        )
        
    
    def forward(self,x):
        x1=x[:,[0,1,2,4,28,31]]
        x2=x[:,[3,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,29,30,32,33]]

        out1=self.module1(x1)        #특정 4개의 열 데이터는 module1에 통과 -> output1 나온다      -shape:[batch_size,16]
        out2=self.module2(x2)        #나머지 13개의 열 데이터는 module2에 통과 -> output2 나온다   -shape:[batch_size,16]

        out=torch.cat([out1,out2],dim=1)       #-shape:[batch_size,32]
        k=out
        residual=k          #초기 결과를 k로 잔차 저장
        for layer in self.hidden_layers:
            j=layer(k)
            k=j+residual
            
        output=self.output_layers(k)
        return output
        


In [10]:
import torch
import numpy as np
import random

def set_seed(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
set_seed(1)

def load_best_model(model, device=device):
    model = model.to(device)
    model.load_state_dict(torch.load("best_model.pt", map_location=device))
    model.eval()
    return model



In [11]:
def scoring(state_input):
    
    X_patient_t = state_input.float()
    if X_patient_t.ndim == 1:
        X_patient_t = X_patient_t.unsqueeze(0) 

    # 모델 불러오기
    model = load_best_model(TotalModel(0.35, 0.35, 1))
    model.eval()
    
    with torch.no_grad():
        output = model(X_patient_t.to(device))   # [1, 3]
        prob = torch.softmax(output, dim=1)      # [1,3]
        prob = prob.squeeze(0)                    # [3]

    # scoring 계산
    score = (prob[1].item() + prob[2].item() * 2) * 50
    return score

In [12]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np


# ======= 1. 환경 정의 =======
class PatientEnv:
    def __init__(self, patient_data, score_model, max_steps=5):
        """
        patient_data: dict {patient_id: {"features": np.array, "label": int}}
        score_model: scoring DNN 모델 (reward용)
        """
        self.patient_data = patient_data
        self.score_model = score_model
        self.max_steps = max_steps
        self.num_features = list(patient_data.values())[0]['features'].shape[0]

    def reset(self, patient_id):  #pid를 받고 해당 환자의 변수정보를 state에 저장
        self.patient = self.patient_data[patient_id]
        self.state = self.patient['features'].copy()
        self.steps = 0
        self.done = False
        return self.state.copy()

    def step(self, action_idx, delta, alpha=5):
        """
        action: [action_idx; 선택할 변수 인덱스 , delta; idx 변수에 대해서 변화시킬 값의 정도] 
        delta: float, 선택 변수에 더할 값
        """


        # 변수별 min-max 범위 적용
        var_min, var_max =col_minmax[action_idx.item()]  # 필요하면 실제 min-max 값 사용

        # reward 계산: scoring model 통과
        old_state_tensor = torch.tensor(self.state, dtype=torch.float32).unsqueeze(0)
        with torch.no_grad():
            old_reward = self.score_model(old_state_tensor)  #이전 시기의 state -> reward계산
        
        
        self.state[action_idx.item()]= np.clip(self.state[action_idx.item()]+ delta.item(), var_min//3, var_max//3)  # action으로 state update

        new_state_tensor = torch.tensor(self.state, dtype=torch.float32).unsqueeze(0)  #new state -> reward계산
        with torch.no_grad():
            new_reward = self.score_model(new_state_tensor)

        self.steps += 1
        self.done = self.steps >= self.max_steps
        return self.state.copy(), alpha*(old_reward-new_reward), self.done   #업데이트된 state 반환, dbs score의 감소량에 비례하는 보상 -> dbs score감소를 촉진, episode 끝났는지 반환

    def state_dim(self):
        return self.num_features

    def action_dim(self):
        return self.num_features  # 34개 변수 중 1개 선택

In [13]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Categorical



class HybridActor(nn.Module):
    def __init__(self, state_dim=34, discrete_dim=34):
        super().__init__()
        self.state_dim = state_dim
        self.discrete_dim = discrete_dim

        # Shared encoder
        self.shared = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU()
        )

        # Discrete head (variable index 선택)
        self.discrete_head = nn.Linear(64, discrete_dim)
        

        # Delta head: state + one-hot(index) 입력
        # input dim = state_dim + discrete_dim
        self.delta_net = nn.Sequential(
            nn.Linear(state_dim + discrete_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 40),
            nn.ReLU(),
            nn.Linear(40, 8),
            nn.ReLU(),
            nn.Linear(8, 3),
            nn.ReLU(),
            nn.Linear(3, 1),
        )

    def forward(self, state):
        h = self.shared(state)
        logits = self.discrete_head(h)
        mask = torch.zeros_like(logits)
        mask[:9] = -1e9
        logits = logits + mask
        dist = Categorical(logits=logits) 
        action_index = dist.sample()   #34개의 변수에 대한 logit값을 얻은 후에, 이 logit으로 dist 만들어서 index 정수값 샘플링한다. (후반에 별로 -> epsilon-greedy?)
        return action_index

    def compute_delta(self, state, action_index):
        if state.dim() == 1:
            state = state.unsqueeze(0)  # [1, state_dim]

        # action_index도 1차원으로 맞춤
        if action_index.dim() == 0:   # 스칼라
            action_index = action_index.unsqueeze(0)  # [1]
        # One-hot 인코딩
        onehot = F.one_hot(action_index.long(), num_classes=self.discrete_dim).float()

        # concat
        x = torch.cat([state, onehot], dim=1)

        # delta 예측
        delta = self.delta_net(x)
        var_min, var_max =col_minmax[action_index.item()]
        delta = torch.clamp(delta, min=var_min//3, max=var_max//3)
        return delta


class HybridCritic(nn.Module):
    def __init__(self, state_dim=34, discrete_dim=34):
        super().__init__()
        self.discrete_dim=discrete_dim
        
        # Input dim = state + onehot(index) + delta
        input_dim = state_dim + discrete_dim + 1
        
        self.q_net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 40),
            nn.ReLU(),
            nn.Linear(40, 8),
            nn.ReLU(),
            nn.Linear(8, 3),
            nn.ReLU(),
            nn.Linear(3, 1),
        )

    def forward(self, state, action_index, delta):
        # action_index : [B] long
        # delta : [B, 1] continuous

        onehot = F.one_hot(action_index, num_classes=self.discrete_dim).float()
        if state.ndim == 1:
            state = state.unsqueeze(0)  # [1, state_dim]
        if onehot.ndim == 1:
            onehot = onehot.unsqueeze(0)  # [1, discrete_dim]

        # delta
        if delta.ndim == 0:
            delta = delta.unsqueeze(0).unsqueeze(1)  # [1,1]
        elif delta.ndim == 1:
            delta = delta.unsqueeze(1)  # [batch, 1]
        x = torch.cat([state, onehot, delta], dim=1)
        q = self.q_net(x)
        return q


In [14]:
def train_ppo(env, actor, critic, epochs=10, gamma=0.99, lr=1e-3,alpha=5):
    actor_opt = optim.Adam(actor.parameters(), lr=lr)
    critic_opt = optim.Adam(critic.parameters(), lr=lr)

    for epoch in range(epochs):
        for pid in env.patient_data.keys():
            state = env.reset(pid)
            done = False

            while not done:
                    state_tensor = torch.tensor(state, dtype=torch.float32)

                    # ============================
                    # 1) ACTOR FORWARD (with grad)
                    # ============================
                    action_idx = actor(state_tensor)
                    delta = actor.compute_delta(state_tensor, action_idx)
                    
                    # ENV STEP
                    next_state, reward, done = env.step(action_idx, delta, alpha=alpha)

                    # ============================
                    # 2) CRITIC UPDATE (with grad)
                    # ============================
                    value = critic(state_tensor, action_idx.detach(), delta.detach())
                    
                    with torch.no_grad():
                        next_value = critic(
                            torch.tensor(next_state, dtype=torch.float32),
                            action_idx.detach(),
                            delta.detach()
                        )
                        target = reward + gamma * next_value * (1 - done)

                    critic_loss = (value - target) ** 2

                    critic_opt.zero_grad()
                    critic_loss.backward()
                    critic_opt.step()

                    # ============================
                    # 3) ACTOR UPDATE (NEW FORWARD!!)
                    # ============================

                    # 새로운 forward (actor-only graph)
                    new_action_idx = actor(state_tensor)
                    new_delta = actor.compute_delta(state_tensor, new_action_idx)

                    actor_loss = -critic(state_tensor, new_action_idx, new_delta)

                    actor_opt.zero_grad()
                    actor_loss.backward()
                    actor_opt.step()

                    state = next_state


In [15]:
# 환자 데이터 로드
num_patients = newdf.shape[0]
num_features = newdf.shape[1]
patient_data = {
    i: {"features": X.iloc[i], "label": y.iloc[i]} for i in range(newdf.shape[0])
}

env = PatientEnv(patient_data, scoring, max_steps=8)
actor = HybridActor()
critic = HybridCritic()

train_ppo(env, actor, critic, epochs=5, lr=3e-4,alpha=10,gamma=0.99)

C:\Users\cmc\AppData\Local\Temp\ipykernel_17296\2455596474.py:11: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  state_tensor = torch.tensor(state, dtype=torch.float32)
C:\Users\cmc\AppData\Local\Temp\ipykernel_17296\1969523763.py:37: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  old_state_tensor = torch.tensor(self.state, dtype=torch.float32).unsqueeze(0)
C:\Users\cmc\AppData\Local\Temp\ipykernel_17296\1969523763.py:42: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value b

In [16]:
torch.save(actor.state_dict(), "actor.pt")

In [17]:
def test_agent(env, actor, pid, max_steps=8,print_mode=True):
    # action index → 문자열 매핑
    feature_names = [
        'gender','age','race','educ','marry','house','pov','wt','ht',
        'bmi','wst','hip','dia','pulse','sys','alt','albumin','ast',
        'crea','chol','tyg','ggt','wbc','hb','hct','ldl','hdl',
        'acratio','glu','insulin','crp','hb1ac','mvpa','ac_week'
    ]

    results = {}

    state = env.reset(pid)
    done = False
    total_reward = 0
    episode_log = []

    for step in range(max_steps):
        state_tensor = torch.tensor(state, dtype=torch.float32)

        # === ACTOR 선택 ===
        action_idx = actor(state_tensor)
        delta = actor.compute_delta(state_tensor, action_idx)

        # === ENV STEP ===
        next_state, reward, done = env.step(action_idx, delta, alpha=10)
        total_reward += reward

        # 기록 저장
        entry = {
            "step": step + 1,
            "action_feature": feature_names[int(action_idx)],
            "action_idx": int(action_idx),
            "delta": float(delta),
            "reward": float(reward)
        }
        episode_log.append(entry)
        if print_mode:
            print(f"Step {entry['step']}:")
            print(f"  Action Feature : {entry['action_feature']}")
            print(f"  Delta          : {entry['delta']:.9f}")
            print(f"  Reward         : {entry['reward']:.9f}")
            print("")  # 줄바꿈

        state = next_state
        if done:
            break

    results[pid] = {
        "total_reward": float(total_reward),
        "episode": episode_log
    }

    return results


In [18]:
test_results = test_agent(env,actor, max_steps=8,pid=762)
print(test_results)


Step 1:
  Action Feature : ast
  Delta          : 3.000000000
  Reward         : -0.000000523

Step 2:
  Action Feature : acratio
  Delta          : 0.000000000
  Reward         : 0.000000000

Step 3:
  Action Feature : ast
  Delta          : 3.000000000
  Reward         : 0.000000000

Step 4:
  Action Feature : acratio
  Delta          : 0.000000000
  Reward         : 0.000000000

Step 5:
  Action Feature : ast
  Delta          : 3.000000000
  Reward         : 0.000000000

Step 6:
  Action Feature : ast
  Delta          : 3.000000000
  Reward         : 0.000000000

Step 7:
  Action Feature : ast
  Delta          : 3.000000000
  Reward         : 0.000000000

Step 8:
  Action Feature : ast
  Delta          : 3.000000000
  Reward         : 0.000000000

{762: {'total_reward': -5.231761690538406e-07, 'episode': [{'step': 1, 'action_feature': 'ast', 'action_idx': 17, 'delta': 3.0, 'reward': -5.231761690538406e-07}, {'step': 2, 'action_feature': 'acratio', 'action_idx': 27, 'delta': 0.0, 're

C:\Users\cmc\AppData\Local\Temp\ipykernel_17296\4190655483.py:18: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  state_tensor = torch.tensor(state, dtype=torch.float32)
C:\Users\cmc\AppData\Local\Temp\ipykernel_17296\1969523763.py:37: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  old_state_tensor = torch.tensor(self.state, dtype=torch.float32).unsqueeze(0)
C:\Users\cmc\AppData\Local\Temp\ipykernel_17296\1969523763.py:42: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value b

Real test

In [19]:

input_dict = {'gender': 1.0, 'age': 33.0, 'race': 1.0, 'educ': 4.0, 'marry': 1.0, 'house': 2.0, 'pov': 3.28, 'wt': 113.8, 
                'ht': 178.5, 'bmi': 35.7, 'wst': 121.8, 'hip': 113.3, 'dia': 79.0, 'pulse': 73.0, 'sys': 114.0, 'alt': 62.0, 'albumin': 4.0, 
                'ast': 32.0, 'crea': 0.82, 'chol': 170.0, 'tyg': 168.0, 'ggt': 36.0, 'wbc': 6.7, 'hb': 15.5, 'hct': 46.3, 'ldl': 106.0, 
                'hdl': 33.0, 'acratio': 5.24, 'glu': 204.0, 'insulin': 22.77, 'crp': 8.81, 'hb1ac': 9.5, 'mvpa': 840.0, 'ac_week': 0.173077, 
                'context':"dd"}


In [20]:
def test_patient(env, actor, patient_dict, max_steps=8):
    feature_names = [
        'gender','age','race','educ','marry','house','pov','wt','ht',
        'bmi','wst','hip','dia','pulse','sys','alt','albumin','ast',
        'crea','chol','tyg','ggt','wbc','hb','hct','ldl','hdl',
        'acratio','glu','insulin','crp','hb1ac','mvpa','ac_week'
    ]

    # state 만들기
    state = np.array([patient_dict[f] for f in feature_names], dtype=np.float32)

    # 결과 dict 초기화
    output_dict = dict(patient_dict)

    old_score = scoring(torch.tensor(state, dtype=torch.float32))


    done = False
    for step in range(1, max_steps + 1):

        state_tensor = torch.tensor(state, dtype=torch.float32)

        # --- action 선택 ---
        action_idx = actor(state_tensor)
        delta = actor.compute_delta(state_tensor, action_idx)
        delta_float=delta.item()

        # === ENV STEP ===
        next_state, reward, done = env.step(action_idx, delta, alpha=5)

        # step 기록
        feature_name = feature_names[int(action_idx.item())]
        output_dict[f"{step}week"] = (feature_name, delta_float)

        state = next_state
        new_score = scoring(torch.tensor(state, dtype=torch.float32))



    output_dict["old_score"] = old_score
    output_dict["new_score"] = new_score

    return output_dict


In [21]:
result = test_patient(env, actor, input_dict)
print(result)

C:\Users\cmc\AppData\Local\Temp\ipykernel_17296\1969523763.py:37: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  old_state_tensor = torch.tensor(self.state, dtype=torch.float32).unsqueeze(0)
C:\Users\cmc\AppData\Local\Temp\ipykernel_17296\1969523763.py:42: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  self.state[action_idx.item()]= np.clip(self.state[action_idx.item()]+ delta.item(), var_min//3, var_max//3)  # action으로 state update
C:\Users\cmc\AppData\Local\Temp\ipykernel_17296\1969523763.py:42: FutureWarning: Series.__setitem__ treating keys as positions is deprecated. In a future version, integer keys will always b

{'gender': 1.0, 'age': 33.0, 'race': 1.0, 'educ': 4.0, 'marry': 1.0, 'house': 2.0, 'pov': 3.28, 'wt': 113.8, 'ht': 178.5, 'bmi': 35.7, 'wst': 121.8, 'hip': 113.3, 'dia': 79.0, 'pulse': 73.0, 'sys': 114.0, 'alt': 62.0, 'albumin': 4.0, 'ast': 32.0, 'crea': 0.82, 'chol': 170.0, 'tyg': 168.0, 'ggt': 36.0, 'wbc': 6.7, 'hb': 15.5, 'hct': 46.3, 'ldl': 106.0, 'hdl': 33.0, 'acratio': 5.24, 'glu': 204.0, 'insulin': 22.77, 'crp': 8.81, 'hb1ac': 9.5, 'mvpa': 840.0, 'ac_week': 0.173077, 'context': 'dd', '1week': ('ast', 3.0), '2week': ('ast', 3.0), '3week': ('ast', 3.0), '4week': ('ast', 3.0), '5week': ('ast', 3.0), '6week': ('ast', 3.0), '7week': ('ast', 3.0), '8week': ('acratio', 0.0), 'old_score': 99.99999342074517, 'new_score': 99.99999347306279}


In [22]:
result

{'gender': 1.0,
 'age': 33.0,
 'race': 1.0,
 'educ': 4.0,
 'marry': 1.0,
 'house': 2.0,
 'pov': 3.28,
 'wt': 113.8,
 'ht': 178.5,
 'bmi': 35.7,
 'wst': 121.8,
 'hip': 113.3,
 'dia': 79.0,
 'pulse': 73.0,
 'sys': 114.0,
 'alt': 62.0,
 'albumin': 4.0,
 'ast': 32.0,
 'crea': 0.82,
 'chol': 170.0,
 'tyg': 168.0,
 'ggt': 36.0,
 'wbc': 6.7,
 'hb': 15.5,
 'hct': 46.3,
 'ldl': 106.0,
 'hdl': 33.0,
 'acratio': 5.24,
 'glu': 204.0,
 'insulin': 22.77,
 'crp': 8.81,
 'hb1ac': 9.5,
 'mvpa': 840.0,
 'ac_week': 0.173077,
 'context': 'dd',
 '1week': ('ast', 3.0),
 '2week': ('ast', 3.0),
 '3week': ('ast', 3.0),
 '4week': ('ast', 3.0),
 '5week': ('ast', 3.0),
 '6week': ('ast', 3.0),
 '7week': ('ast', 3.0),
 '8week': ('acratio', 0.0),
 'old_score': 99.99999342074517,
 'new_score': 99.99999347306279}

In [23]:

actor2 = HybridActor()
actor2.load_state_dict(torch.load("actor.pt"))
actor2.eval()

# 3) 평가 실행
rl_output = test_patient(env, actor2, input_dict)
print(rl_output)

C:\Users\cmc\AppData\Local\Temp\ipykernel_17296\1969523763.py:37: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  old_state_tensor = torch.tensor(self.state, dtype=torch.float32).unsqueeze(0)


{'gender': 1.0, 'age': 33.0, 'race': 1.0, 'educ': 4.0, 'marry': 1.0, 'house': 2.0, 'pov': 3.28, 'wt': 113.8, 'ht': 178.5, 'bmi': 35.7, 'wst': 121.8, 'hip': 113.3, 'dia': 79.0, 'pulse': 73.0, 'sys': 114.0, 'alt': 62.0, 'albumin': 4.0, 'ast': 32.0, 'crea': 0.82, 'chol': 170.0, 'tyg': 168.0, 'ggt': 36.0, 'wbc': 6.7, 'hb': 15.5, 'hct': 46.3, 'ldl': 106.0, 'hdl': 33.0, 'acratio': 5.24, 'glu': 204.0, 'insulin': 22.77, 'crp': 8.81, 'hb1ac': 9.5, 'mvpa': 840.0, 'ac_week': 0.173077, 'context': 'dd', '1week': ('ast', 3.0), '2week': ('ast', 3.0), '3week': ('ast', 3.0), '4week': ('ast', 3.0), '5week': ('ast', 3.0), '6week': ('ast', 3.0), '7week': ('ast', 3.0), '8week': ('ast', 3.0), 'old_score': 99.99999342074517, 'new_score': 99.99999347306279}


C:\Users\cmc\AppData\Local\Temp\ipykernel_17296\1969523763.py:42: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  self.state[action_idx.item()]= np.clip(self.state[action_idx.item()]+ delta.item(), var_min//3, var_max//3)  # action으로 state update
C:\Users\cmc\AppData\Local\Temp\ipykernel_17296\1969523763.py:42: FutureWarning: Series.__setitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To set a value by position, use `ser.iloc[pos] = value`
  self.state[action_idx.item()]= np.clip(self.state[action_idx.item()]+ delta.item(), var_min//3, var_max//3)  # action으로 state update
C:\Users\cmc\AppData\Local\Temp\ipykernel_17296\1969523763.py:44: FutureWarning: Series.__getitem__ treating keys as positions is